# Notebook 03 — Fixing class imbalance

**Goal:** Apply two techniques to improve minority class scores:
1. `class_weight='balanced'` — tells the model to penalise minority class errors more
2. SMOTE — creates synthetic minority samples before training

We compare all three approaches (baseline, class weights, SMOTE) side by side.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.load_data import load_processed
from src.preprocess import get_X_y, ENDANGERMENT_ORDER
from src.train import split_data, train_class_weights, train_smote, load_model
from src.evaluate import evaluate_model, print_classification_report, compare_models
from src.visualise import plot_confusion_matrix, plot_f1_comparison

print('Imports OK')

In [ ]:
df = load_processed()
X, y = get_X_y(df)
X_train, X_test, y_train, y_test = split_data(X, y)

In [ ]:
# ── Approach 1: class weights ─────────────────────────────────────────
# The simplest fix. One argument: class_weight='balanced'
# sklearn automatically computes a weight for each class:
#   weight = total_samples / (n_classes * class_count)
# Minority classes get high weights → model pays more attention to them.

model_cw = train_class_weights(X_train, y_train)
results_cw = evaluate_model(model_cw, X_test, y_test, ENDANGERMENT_ORDER)
print_classification_report(results_cw)

In [ ]:
plot_confusion_matrix(
    results_cw['cm'],
    results_cw['cm_labels'],
    title='Class-weighted model — confusion matrix',
    normalise=True,
    save_name='03_cm_class_weights'
)

In [ ]:
# ── Approach 2: SMOTE ────────────────────────────────────────────────
# SMOTE = Synthetic Minority Over-sampling Technique
# Creates new synthetic training samples for minority classes by interpolating
# between real minority class samples.
#
# CRITICAL: SMOTE is applied ONLY to training data.
# Never apply it to test data.

model_smote, _, _ = train_smote(X_train, y_train)
results_smote = evaluate_model(model_smote, X_test, y_test, ENDANGERMENT_ORDER)
print_classification_report(results_smote)

In [ ]:
plot_confusion_matrix(
    results_smote['cm'],
    results_smote['cm_labels'],
    title='SMOTE model — confusion matrix',
    normalise=True,
    save_name='03_cm_smote'
)

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────
# Load baseline results from notebook 02 model
model_base = load_model('baseline')
results_base = evaluate_model(model_base, X_test, y_test, ENDANGERMENT_ORDER)

all_results = {
    'Baseline': results_base,
    'Class weights': results_cw,
    'SMOTE': results_smote,
}
comparison = compare_models(all_results)
print(comparison.to_string())

In [ ]:
# ── F1 comparison bar chart ────────────────────────────────────────────
# This plot shows which model improved minority class F1 scores.
# Ideal: the bars for Critically Endangered and Extinct should be taller
# for SMOTE and Class Weights than for Baseline.
plot_f1_comparison(comparison, save_name='03_f1_comparison')

In [ ]:
print('=== KEY TAKEAWAY FROM NOTEBOOK 03 ===')
print(f'Baseline macro F1:      {results_base["macro_f1"]:.3f}')
print(f'Class weights macro F1: {results_cw["macro_f1"]:.3f}')
print(f'SMOTE macro F1:         {results_smote["macro_f1"]:.3f}')
print()
print('The technique that improves macro F1 most is improving')
print('the minority class recall — verified by the confusion matrix.')
print()
print('Proceed to 04_confusion_matrix.ipynb for a deep dive.')